# Scale vs Configuration: Phase 2 technical canary v4

This revision has **one executable cell**. It preserves the frozen scientific protocol and runner while removing cross-cell notebook state and the broken v3 Git peel expression.

Before running:

1. Select **Runtime > Change runtime type > Runtime Version 2026.07 > GPU**.
2. Confirm that `phase2_stimuli_transport_v1.tar` remains in `MyDrive/scale-vs-configuration/phase2/artifacts/`.
3. In the single code cell, change `REQUESTED_MODEL_KEY` from `SELECT_MODEL` to exactly one model.
4. Run the code cell once using its play button. Do not use `Run all`, reconnect, restart, or rerun the cell.

The cell creates a unique v4 attempt directory, runs the blocking preflight, executes one query, resumes to all 12 queries in the same process, and writes the technical validation and collection manifest. If any stage fails, the same cell stops and records `technical_failure.json`.

This notebook contains no full-inference command. V2 and v3 directories are preserved but never read or written by v4. Technical canary outputs remain excluded from scientific analysis.

In [ ]:
# SINGLE EXECUTABLE CELL: internally performs Steps 1 through 5 in order.
REQUESTED_MODEL_KEY = "SELECT_MODEL"  # @param ["SELECT_MODEL", "qwen3_vl_4b_instruct", "internvl3_5_4b_hf", "llava_next_mistral_7b"]

from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import tarfile
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

CANARY_GIT_REF = "phase2-canary-v4"
REPOSITORY_URL = "https://github.com/grifo114/scale-vs-configuration.git"
ALLOWED_MODEL_KEYS = {
    "qwen3_vl_4b_instruct",
    "internvl3_5_4b_hf",
    "llava_next_mistral_7b",
}

if REQUESTED_MODEL_KEY not in ALLOWED_MODEL_KEYS:
    raise RuntimeError("Select one model in Step 1 before running this cell.")

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def run_checked(command, *, cwd=None, env=None, log_path=None):
    command = [str(item) for item in command]
    completed = subprocess.run(
        command, cwd=cwd, env=env, text=True, capture_output=True
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if log_path is not None:
        Path(log_path).write_text(
            "COMMAND\n" + " ".join(command)
            + "\n\nSTDOUT\n" + completed.stdout
            + "\nSTDERR\n" + completed.stderr,
            encoding="utf-8",
        )
    completed.check_returncode()
    return completed

if "_V4_CONTEXT" in globals():
    if _V4_CONTEXT["model_key"] != REQUESTED_MODEL_KEY:
        raise RuntimeError(
            "STEP 1 ALREADY INITIALIZED FOR ANOTHER MODEL. "
            "Start a fresh Colab runtime; do not change models in this runtime."
        )
    print("STEP 1 was already initialized in this kernel. Existing context retained.")
else:
    MODEL_KEY = REQUESTED_MODEL_KEY
    SESSION_TOKEN = uuid.uuid4().hex
    RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "_" + uuid.uuid4().hex[:8]
    )

    DRIVE_ROOT = Path("/content/drive/MyDrive/scale-vs-configuration/phase2")
    DRIVE_ARCHIVE = DRIVE_ROOT / "artifacts/phase2_stimuli_transport_v1.tar"
    ATTEMPT_ROOT = DRIVE_ROOT / "v4_attempts" / MODEL_KEY / RUN_ID
    OUTPUT_ROOT = ATTEMPT_ROOT / "inference"
    MODEL_OUTPUT_DIR = OUTPUT_ROOT / MODEL_KEY / "canary"
    PROVENANCE_DIR = ATTEMPT_ROOT / "provenance"
    CACHE_DIR = DRIVE_ROOT / "hf_cache" / MODEL_KEY
    STATE_PATH = ATTEMPT_ROOT / "attempt_state.json"
    LOCAL_ARCHIVE = Path("/content/phase2_stimuli_transport_v1.tar")
    REPO_ROOT = Path("/content/scale-vs-configuration-v4")

    if ATTEMPT_ROOT.exists():
        raise RuntimeError(f"Unique attempt path unexpectedly exists: {ATTEMPT_ROOT}")

    for directory in (OUTPUT_ROOT, PROVENANCE_DIR, CACHE_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    _V4_CONTEXT = {
        "attempt_root": str(ATTEMPT_ROOT),
        "cache_dir": str(CACHE_DIR),
        "model_key": MODEL_KEY,
        "model_output_dir": str(MODEL_OUTPUT_DIR),
        "output_root": str(OUTPUT_ROOT),
        "provenance_dir": str(PROVENANCE_DIR),
        "run_id": RUN_ID,
        "session_token": SESSION_TOKEN,
        "state_path": str(STATE_PATH),
    }

    write_json(STATE_PATH, {
        "schema_version": "1.0",
        "artifact_name": "phase2_colab_v4_attempt_state",
        "attempt_id": RUN_ID,
        "canary_git_ref": CANARY_GIT_REF,
        "created_at_utc": utc_now(),
        "kernel_pid": os.getpid(),
        "model_key": MODEL_KEY,
        "notebook_version": "v4",
        "session_hostname": socket.gethostname(),
        "session_token": SESSION_TOKEN,
        "status": "initialized",
    })

def require_context(expected_status):
    if "_V4_CONTEXT" not in globals():
        raise RuntimeError(
            "V4 SESSION STATE IS ABSENT. Do not resume an existing attempt. "
            "Start Step 1 in a fresh runtime to create a new unique attempt."
        )
    state_path = Path(_V4_CONTEXT["state_path"])
    state = json.loads(state_path.read_text(encoding="utf-8"))
    if state["session_token"] != _V4_CONTEXT["session_token"]:
        raise RuntimeError("Persistent and in-memory session tokens differ.")
    if state["model_key"] != _V4_CONTEXT["model_key"]:
        raise RuntimeError("Persistent and in-memory model keys differ.")
    if state["status"] != expected_status:
        raise RuntimeError(
            f"STEP ORDER ERROR: expected state {expected_status!r}, "
            f"found {state['status']!r}. Do not rerun completed steps."
        )
    return state

def update_state(expected_status, new_status, **extra):
    state = require_context(expected_status)
    state.update(extra)
    state["status"] = new_status
    state["updated_at_utc"] = utc_now()
    write_json(Path(_V4_CONTEXT["state_path"]), state)

def record_failure(expected_status, stage, error):
    failure = {
        "schema_version": "1.0",
        "attempt_id": _V4_CONTEXT["run_id"],
        "captured_at_utc": utc_now(),
        "error_message": str(error),
        "error_type": type(error).__name__,
        "model_key": _V4_CONTEXT["model_key"],
        "stage": stage,
        "traceback": traceback.format_exc(),
    }
    write_json(Path(_V4_CONTEXT["provenance_dir"]) / "technical_failure.json", failure)
    update_state(expected_status, "failed", failed_stage=stage)

state = require_context("initialized")
print("\n===== V4 ATTEMPT INITIALIZED =====")
print(f"Model: {_V4_CONTEXT['model_key']}")
print(f"Attempt ID: {_V4_CONTEXT['run_id']}")
print(f"Attempt root: {_V4_CONTEXT['attempt_root']}")
print("State: initialized")
print("Continuing automatically to Step 2.")

# STEP 2 OF 5: complete blocking preflight. No model inference occurs here.
if "_V4_CONTEXT" not in globals():
    raise RuntimeError("V4 SESSION STATE IS ABSENT. Start a new attempt at Step 1.")

require_context("initialized")
MODEL_KEY = _V4_CONTEXT["model_key"]
ATTEMPT_ROOT = Path(_V4_CONTEXT["attempt_root"])
OUTPUT_ROOT = Path(_V4_CONTEXT["output_root"])
MODEL_OUTPUT_DIR = Path(_V4_CONTEXT["model_output_dir"])
PROVENANCE_DIR = Path(_V4_CONTEXT["provenance_dir"])
CACHE_DIR = Path(_V4_CONTEXT["cache_dir"])
STATE_PATH = Path(_V4_CONTEXT["state_path"])
DRIVE_ROOT = Path("/content/drive/MyDrive/scale-vs-configuration/phase2")
DRIVE_ARCHIVE = DRIVE_ROOT / "artifacts/phase2_stimuli_transport_v1.tar"
LOCAL_ARCHIVE = Path("/content/phase2_stimuli_transport_v1.tar")
REPO_ROOT = Path("/content/scale-vs-configuration-v4")

try:
    import numpy as np
    import torch

    os_release = {}
    for line in Path("/etc/os-release").read_text(encoding="utf-8").splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os_release[key] = value.strip().strip('\"')

    base_runtime = {
        "captured_at_utc": utc_now(),
        "declared_colab_runtime_version": "2026.07",
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "ubuntu": os_release.get("PRETTY_NAME"),
        "cuda_available": bool(torch.cuda.is_available()),
        "torch_cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "gpu_devices": [],
    }
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            base_runtime["gpu_devices"].append({
                "index": index,
                "name": props.name,
                "total_memory_bytes": props.total_memory,
                "compute_capability": [props.major, props.minor],
            })

    assert base_runtime["python"] == "3.12.13", "Select Colab runtime 2026.07"
    assert base_runtime["torch"].split("+", 1)[0] == "2.11.0"
    assert base_runtime["numpy"] == "2.0.2"
    assert base_runtime["ubuntu"] == "Ubuntu 22.04.5 LTS"
    assert base_runtime["cuda_available"] is True
    assert len(base_runtime["gpu_devices"]) >= 1
    write_json(PROVENANCE_DIR / "base_runtime.json", base_runtime)
    print("Base Colab runtime preflight: OK")

    if REPO_ROOT.exists():
        raise RuntimeError(
            f"Repository path already exists in this attempt runtime: {REPO_ROOT}"
        )
    run_checked([
        "git", "clone", "--branch", CANARY_GIT_REF, "--depth", "1",
        REPOSITORY_URL, str(REPO_ROOT),
    ])
    remote_url = run_checked(
        ["git", "config", "--get", "remote.origin.url"], cwd=REPO_ROOT
    ).stdout.strip()
    if remote_url not in {REPOSITORY_URL, REPOSITORY_URL.removesuffix(".git")}:
        raise RuntimeError(f"Unexpected repository remote: {remote_url}")
    head = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).stdout.strip()
    ref_commit = run_checked(
        ["git", "rev-parse", f"{CANARY_GIT_REF}^{{commit}}"], cwd=REPO_ROOT
    ).stdout.strip()
    assert head == ref_commit
    assert run_checked(
        ["git", "status", "--porcelain=v1", "--untracked-files=no"], cwd=REPO_ROOT
    ).stdout == ""

    expected_hashes = {
        "configs/phase2_inference_protocol.json": "3c09ca97984f47b868758ce3b4d07bf85e38dabd3d66b7e4eae9bb474ce4b774",
        "configs/phase2_query_manifest.jsonl": "61e4f9a8583dc6f375b6d4d9f72361acf1d9ff3a976485bf5b2f83736ea0360e",
        "scripts/run_phase2_inference.py": "643a63625d97aa893322410cf865bd2aa190d2f9b5d4c984aa6c50ef09a38818",
        "scripts/package_phase2_stimuli.py": "aae1366fc05caf13a410676c7cea88df417cbeb9c82a3825288d887cba09b185",
        "configs/phase2_stimuli_transport_v1.json": "c3e88b6225d7026005c6419830fcb3f3ffda3d39ec64e0986c3e15cdac6f8fb0",
        "configs/phase2_colab_environment_v1.json": "4d2e96152a71a0105c2cdbf1ec817d1f3eaf75ed4ff98b1f05bb071deab9045c",
        "configs/phase2_colab_execution_revision_v4.json": "09175329b35d13ddd768d02578541e59fc2c16b2033f064ad7ab5e77df53763a",
    }
    for relative_path, expected in expected_hashes.items():
        actual = sha256_file(REPO_ROOT / relative_path)
        print(f"{relative_path}: {actual}")
        assert actual == expected

    environment_spec = json.loads(
        (REPO_ROOT / "configs/phase2_colab_environment_v1.json").read_text(encoding="utf-8")
    )
    protocol = json.loads(
        (REPO_ROOT / "configs/phase2_inference_protocol.json").read_text(encoding="utf-8")
    )
    execution_revision = json.loads(
        (REPO_ROOT / "configs/phase2_colab_execution_revision_v4.json").read_text(encoding="utf-8")
    )
    assert environment_spec["status"] == "predeclared_pending_colab_preflight"
    assert environment_spec["scientific_design_changed"] is False
    assert environment_spec["canary_execution"]["full_mode_allowed"] is False
    assert protocol["protocol_status"] == "predeclared_pending_canary"
    assert execution_revision["scientific_design_changed"] is False
    assert execution_revision["scientific_outputs_analyzed"] is False
    assert execution_revision["full_inference_allowed"] is False

    write_json(PROVENANCE_DIR / "git_identity.json", {
        "attempt_id": _V4_CONTEXT["run_id"],
        "captured_at_utc": utc_now(),
        "remote_url": remote_url,
        "requested_ref": CANARY_GIT_REF,
        "resolved_commit": head,
    })
    print("Frozen Git inputs: OK")

    pins = environment_spec["pip_install"]["packages_in_install_order"]
    assert environment_spec["pip_install"]["upgrade_strategy"] == "only-if-needed"
    assert environment_spec["pip_install"]["allow_prereleases"] is False
    run_checked([
        sys.executable, "-m", "pip", "install",
        "--upgrade-strategy", "only-if-needed", *pins,
    ], log_path=PROVENANCE_DIR / "pip_install.log")

    import importlib
    import importlib.metadata
    global_pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
    )
    pip_check_text = "\n".join(
        part.rstrip() for part in (
            global_pip_check.stdout, global_pip_check.stderr
        ) if part.strip()
    )
    if pip_check_text:
        pip_check_text += "\n"
    (PROVENANCE_DIR / "pip_check.log").write_text(pip_check_text, encoding="utf-8")
    print(f"Global pip check return code: {global_pip_check.returncode}")
    if pip_check_text:
        print(pip_check_text, end="")
    if global_pip_check.returncode != 0:
        print("Global pip check conflicts recorded as diagnostic-only.")

    expected_versions = dict(item.rsplit("==", 1) for item in pins)
    targeted_modules = {
        "huggingface-hub": "huggingface_hub",
        "tokenizers": "tokenizers",
        "safetensors": "safetensors",
        "sentencepiece": "sentencepiece",
        "pillow": "PIL",
        "accelerate": "accelerate",
        "bitsandbytes": "bitsandbytes",
        "transformers": "transformers",
        "numpy": "numpy",
        "torch": "torch",
    }
    targeted_import_results = {}
    for distribution, module_name in targeted_modules.items():
        importlib.import_module(module_name)
        targeted_import_results[distribution] = {"module": module_name, "status": "ok"}
    installed_versions = {
        package: importlib.metadata.version(package) for package in expected_versions
    }
    for package, expected in expected_versions.items():
        actual = installed_versions[package]
        print(f"{package}: {actual}")
        assert actual == expected
    assert importlib.metadata.version("numpy") == "2.0.2"
    assert importlib.metadata.version("torch").split("+", 1)[0] == "2.11.0"
    write_json(PROVENANCE_DIR / "pip_check_report.json", {
        "schema_version": "1.0",
        "blocking": False,
        "clean": global_pip_check.returncode == 0,
        "pinned_package_versions": installed_versions,
        "returncode": global_pip_check.returncode,
        "scope": "global_colab_environment_diagnostic_only",
        "targeted_imports": targeted_import_results,
        "targeted_imports_completed": True,
    })
    write_json(PROVENANCE_DIR / "pinned_package_versions.json", installed_versions)
    print("Targeted pinned package environment: OK")

    transport = environment_spec["stimulus_transport"]
    assert DRIVE_ARCHIVE.is_file(), f"Archive absent: {DRIVE_ARCHIVE}"
    assert DRIVE_ARCHIVE.stat().st_size == transport["archive_size_bytes"]
    drive_archive_sha = sha256_file(DRIVE_ARCHIVE)
    assert drive_archive_sha == transport["archive_sha256"]
    if not LOCAL_ARCHIVE.is_file() or sha256_file(LOCAL_ARCHIVE) != transport["archive_sha256"]:
        shutil.copyfile(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
    assert LOCAL_ARCHIVE.stat().st_size == transport["archive_size_bytes"]
    local_archive_sha = sha256_file(LOCAL_ARCHIVE)
    assert local_archive_sha == transport["archive_sha256"]

    manifest_rows = [
        json.loads(line)
        for line in (REPO_ROOT / "configs/phase2_query_manifest.jsonl")
            .read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    manifest_rows.sort(key=lambda row: int(row["run_index"]))
    expected_member_names = [str(row["stimulus_path"]) for row in manifest_rows]
    assert len(expected_member_names) == 588
    assert len(set(expected_member_names)) == 588
    existing_targets = [(REPO_ROOT / name).exists() for name in expected_member_names]
    if any(existing_targets) and not all(existing_targets):
        raise RuntimeError("Partial stimulus extraction detected")
    with tarfile.open(LOCAL_ARCHIVE, mode="r:") as archive:
        members = archive.getmembers()
        assert len(members) == transport["entry_count"] == 588
        assert [member.name for member in members] == expected_member_names
        for member in members:
            pure = PurePosixPath(member.name)
            assert member.isfile()
            assert not pure.is_absolute()
            assert ".." not in pure.parts
        if not any(existing_targets):
            archive.extractall(path=REPO_ROOT, members=members, filter="data")
    assert all((REPO_ROOT / name).is_file() for name in expected_member_names)
    write_json(PROVENANCE_DIR / "stimulus_transport_verification.json", {
        "attempt_id": _V4_CONTEXT["run_id"],
        "captured_at_utc": utc_now(),
        "drive_archive_path": str(DRIVE_ARCHIVE),
        "drive_archive_sha256": drive_archive_sha,
        "entry_count": len(expected_member_names),
        "local_archive_path": str(LOCAL_ARCHIVE),
        "local_archive_sha256": local_archive_sha,
    })
    print("Safe stimulus extraction: OK")

    runner_path = REPO_ROOT / "scripts/run_phase2_inference.py"
    run_checked(
        [sys.executable, str(runner_path), "--project-root", str(REPO_ROOT), "--validate-only"],
        cwd=REPO_ROOT,
        log_path=PROVENANCE_DIR / "runner_validate_only.log",
    )
    print("Runner validation of all 588 stimuli: OK")

    from huggingface_hub import HfApi
    api = HfApi()
    revision_checks = []
    for model in protocol["models"]:
        info = api.model_info(model["model_id"], revision=model["revision"])
        check = {
            "expected_revision": model["revision"],
            "matches": info.sha == model["revision"],
            "model_id": model["model_id"],
            "model_key": model["key"],
            "resolved_revision": info.sha,
        }
        revision_checks.append(check)
        assert check["matches"] is True
    write_json(PROVENANCE_DIR / "frozen_revision_checks.json", {
        "attempt_id": _V4_CONTEXT["run_id"],
        "captured_at_utc": utc_now(),
        "checks": revision_checks,
    })
    print("All three frozen model revisions resolve: OK")

    pip_freeze = run_checked([sys.executable, "-m", "pip", "freeze", "--all"]).stdout
    (PROVENANCE_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")
    selected_model = next(model for model in protocol["models"] if model["key"] == MODEL_KEY)
    preflight = {
        "schema_version": "1.0",
        "all_frozen_revisions_resolved": all(item["matches"] for item in revision_checks),
        "attempt_id": _V4_CONTEXT["run_id"],
        "base_runtime": base_runtime,
        "captured_at_utc": utc_now(),
        "full_mode_allowed": False,
        "git_commit": head,
        "git_ref": CANARY_GIT_REF,
        "global_pip_check_blocking": False,
        "global_pip_check_returncode": global_pip_check.returncode,
        "model_id": selected_model["model_id"],
        "model_key": MODEL_KEY,
        "model_revision": selected_model["revision"],
        "pinned_package_versions": installed_versions,
        "runner_validate_only_completed": True,
        "session_token": _V4_CONTEXT["session_token"],
        "stimulus_archive_sha256": local_archive_sha,
        "targeted_imports_completed": True,
    }
    write_json(PROVENANCE_DIR / "colab_preflight.json", preflight)
    update_state(
        "initialized", "preflight_complete",
        git_commit=head,
        model_id=selected_model["model_id"],
        model_revision=selected_model["revision"],
    )
    print("\n===== STEP 2 COMPLETE =====")
    print("Persistent preflight capture: OK")
    print("Continuing automatically to Step 3.")
except Exception as error:
    record_failure("initialized", "preflight", error)
    raise

# STEP 3 OF 5: execute exactly one query.
if "_V4_CONTEXT" not in globals():
    raise RuntimeError("V4 SESSION STATE IS ABSENT. Start a new attempt at Step 1.")
require_context("preflight_complete")

MODEL_KEY = _V4_CONTEXT["model_key"]
OUTPUT_ROOT = Path(_V4_CONTEXT["output_root"])
MODEL_OUTPUT_DIR = Path(_V4_CONTEXT["model_output_dir"])
PROVENANCE_DIR = Path(_V4_CONTEXT["provenance_dir"])
CACHE_DIR = Path(_V4_CONTEXT["cache_dir"])
REPO_ROOT = Path("/content/scale-vs-configuration-v4")
runner_path = REPO_ROOT / "scripts/run_phase2_inference.py"
results_path = MODEL_OUTPUT_DIR / "results.jsonl"
metadata_path = MODEL_OUTPUT_DIR / "metadata.json"
failures_path = MODEL_OUTPUT_DIR / "failures.jsonl"

try:
    if results_path.exists() or metadata_path.exists() or failures_path.exists():
        raise RuntimeError("Attempt-specific inference directory is not empty before Step 3")
    runner_environment = os.environ.copy()
    runner_environment["PYTHONHASHSEED"] = "0"
    runner_environment["TOKENIZERS_PARALLELISM"] = "false"
    runner_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    runner_command = [
        sys.executable, str(runner_path),
        "--project-root", str(REPO_ROOT),
        "--model-key", MODEL_KEY,
        "--mode", "canary",
        "--output-root", str(OUTPUT_ROOT),
        "--cache-dir", str(CACHE_DIR),
    ]
    run_checked(
        [*runner_command, "--max-new-queries", "1"],
        cwd=REPO_ROOT, env=runner_environment,
        log_path=PROVENANCE_DIR / "invocation_01_max_new_queries_1.log",
    )
    first_rows = [
        json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    first_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert len(first_rows) == 1
    assert first_metadata["status"] == "partial"
    assert first_metadata["completed_queries"] == 1
    assert not failures_path.exists() or failures_path.stat().st_size == 0
    update_state(
        "preflight_complete", "first_query_complete",
        completed_queries=1,
        runtime_fingerprint_sha256=first_metadata["runtime_fingerprint_sha256"],
    )
    print("\n===== STEP 3 COMPLETE =====")
    print("First invocation completed exactly 1/12 queries: OK")
    print("Continuing automatically to Step 4 in this runtime.")
except Exception as error:
    record_failure("preflight_complete", "first_query", error)
    raise

# STEP 4 OF 5: resume the exact one-row prefix to twelve queries.
if "_V4_CONTEXT" not in globals():
    raise RuntimeError(
        "V4 SESSION STATE IS ABSENT. Same-session resume is not permitted. "
        "Preserve the old attempt and start a new attempt at Step 1."
    )
state = require_context("first_query_complete")

MODEL_KEY = _V4_CONTEXT["model_key"]
OUTPUT_ROOT = Path(_V4_CONTEXT["output_root"])
MODEL_OUTPUT_DIR = Path(_V4_CONTEXT["model_output_dir"])
PROVENANCE_DIR = Path(_V4_CONTEXT["provenance_dir"])
CACHE_DIR = Path(_V4_CONTEXT["cache_dir"])
REPO_ROOT = Path("/content/scale-vs-configuration-v4")
runner_path = REPO_ROOT / "scripts/run_phase2_inference.py"
results_path = MODEL_OUTPUT_DIR / "results.jsonl"
metadata_path = MODEL_OUTPUT_DIR / "metadata.json"
failures_path = MODEL_OUTPUT_DIR / "failures.jsonl"

try:
    current_rows = [
        json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert len(current_rows) == 1, f"Expected exact one-row prefix; found {len(current_rows)}"
    current_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert current_metadata["status"] == "partial"
    assert current_metadata["completed_queries"] == 1
    assert current_metadata["runtime_fingerprint_sha256"] == state["runtime_fingerprint_sha256"]
    assert not failures_path.exists() or failures_path.stat().st_size == 0

    runner_environment = os.environ.copy()
    runner_environment["PYTHONHASHSEED"] = "0"
    runner_environment["TOKENIZERS_PARALLELISM"] = "false"
    runner_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    runner_command = [
        sys.executable, str(runner_path),
        "--project-root", str(REPO_ROOT),
        "--model-key", MODEL_KEY,
        "--mode", "canary",
        "--output-root", str(OUTPUT_ROOT),
        "--cache-dir", str(CACHE_DIR),
    ]
    run_checked(
        runner_command, cwd=REPO_ROOT, env=runner_environment,
        log_path=PROVENANCE_DIR / "invocation_02_resume_to_12.log",
    )
    completed_rows = [
        json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    completed_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert len(completed_rows) == 12
    assert completed_metadata["status"] == "complete"
    assert completed_metadata["completed_queries"] == 12
    assert completed_metadata["runtime_fingerprint_sha256"] == state["runtime_fingerprint_sha256"]
    assert not failures_path.exists() or failures_path.stat().st_size == 0
    update_state("first_query_complete", "resume_complete", completed_queries=12)
    print("\n===== STEP 4 COMPLETE =====")
    print("Exact-prefix resumption completed 12/12 canary queries: OK")
    print("Continuing automatically to Step 5.")
except Exception as error:
    record_failure("first_query_complete", "resume_to_twelve", error)
    raise

# STEP 5 OF 5: validate technical evidence and close this unique attempt.
if "_V4_CONTEXT" not in globals():
    raise RuntimeError("V4 SESSION STATE IS ABSENT. Start a new attempt at Step 1.")
state = require_context("resume_complete")

MODEL_KEY = _V4_CONTEXT["model_key"]
ATTEMPT_ROOT = Path(_V4_CONTEXT["attempt_root"])
MODEL_OUTPUT_DIR = Path(_V4_CONTEXT["model_output_dir"])
PROVENANCE_DIR = Path(_V4_CONTEXT["provenance_dir"])
STATE_PATH = Path(_V4_CONTEXT["state_path"])
REPO_ROOT = Path("/content/scale-vs-configuration-v4")
results_path = MODEL_OUTPUT_DIR / "results.jsonl"
metadata_path = MODEL_OUTPUT_DIR / "metadata.json"
failures_path = MODEL_OUTPUT_DIR / "failures.jsonl"

try:
    protocol = json.loads(
        (REPO_ROOT / "configs/phase2_inference_protocol.json").read_text(encoding="utf-8")
    )
    manifest_rows = [
        json.loads(line)
        for line in (REPO_ROOT / "configs/phase2_query_manifest.jsonl")
            .read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    manifest_rows.sort(key=lambda row: int(row["run_index"]))
    canary_rows = sorted(
        [row for row in manifest_rows if row["is_canary"] is True],
        key=lambda row: int(row["canary_index"]),
    )
    completed_rows = [
        json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    completed_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    selected_model = next(model for model in protocol["models"] if model["key"] == MODEL_KEY)
    assert len(canary_rows) == 12
    assert len(completed_rows) == 12

    expected_hashes = {
        "protocol": "3c09ca97984f47b868758ce3b4d07bf85e38dabd3d66b7e4eae9bb474ce4b774",
        "query_manifest": "61e4f9a8583dc6f375b6d4d9f72361acf1d9ff3a976485bf5b2f83736ea0360e",
        "runner": "643a63625d97aa893322410cf865bd2aa190d2f9b5d4c984aa6c50ef09a38818",
    }
    allowed_response_statuses = {"valid", "invalid_json", "invalid_schema"}
    runtime_fingerprints = set()
    for sequence_index, (query, result) in enumerate(zip(canary_rows, completed_rows), start=1):
        assert result["sequence_index"] == sequence_index
        assert result["pair_id"] == query["pair_id"]
        assert result["run_index"] == query["run_index"]
        assert result["canary_index"] == query["canary_index"]
        assert result["scene_id"] == query["scene_id"]
        assert result["fold"] == query["fold"]
        assert result["distance_stratum"] == query["distance_stratum"]
        assert result["stimulus_path"] == query["stimulus_path"]
        assert result["stimulus_sha256"] == query["stimulus_sha256"]
        assert result["prompt"] == query["prompt"]
        assert result["prompt_sha256"] == hashlib.sha256(query["prompt"].encode("utf-8")).hexdigest()
        assert result["ground_truth_m"] == query["ground_truth_m"]
        assert result["model_key"] == MODEL_KEY
        assert result["model_id"] == selected_model["model_id"]
        assert result["model_revision"] == selected_model["revision"]
        assert result["protocol_sha256"] == expected_hashes["protocol"]
        assert result["query_manifest_sha256"] == expected_hashes["query_manifest"]
        assert result["runner_sha256"] == expected_hashes["runner"]
        assert isinstance(result["raw_response"], str)
        assert result["response_status"] in allowed_response_statuses
        runtime_fingerprints.add(result["runtime_fingerprint_sha256"])

    assert runtime_fingerprints == {state["runtime_fingerprint_sha256"]}
    assert runtime_fingerprints == {completed_metadata["runtime_fingerprint_sha256"]}
    assert completed_metadata["mode"] == "canary"
    assert completed_metadata["model_key"] == MODEL_KEY
    assert completed_metadata["model_revision"] == selected_model["revision"]
    assert completed_metadata["git"]["commit"] == state["git_commit"]
    assert completed_metadata["target_queries"] == 12
    assert completed_metadata["valid_responses"] + completed_metadata["invalid_responses"] == 12
    assert not failures_path.exists() or failures_path.stat().st_size == 0

    first_log_path = PROVENANCE_DIR / "invocation_01_max_new_queries_1.log"
    resume_log_path = PROVENANCE_DIR / "invocation_02_resume_to_12.log"
    assert "Existing exact prefix: 0/12" in first_log_path.read_text(encoding="utf-8")
    assert "Existing exact prefix: 1/12" in resume_log_path.read_text(encoding="utf-8")

    validation_report = {
        "schema_version": "1.0",
        "attempt_id": _V4_CONTEXT["run_id"],
        "exact_prefix_resume_verified": True,
        "git_commit": state["git_commit"],
        "git_ref": CANARY_GIT_REF,
        "invalid_responses_retained": completed_metadata["invalid_responses"],
        "model_id": selected_model["model_id"],
        "model_key": MODEL_KEY,
        "model_revision": selected_model["revision"],
        "result_rows": len(completed_rows),
        "runtime_fingerprint_sha256": completed_metadata["runtime_fingerprint_sha256"],
        "scientific_results_analyzed": False,
        "status": "complete_pending_cross_model_review",
        "technical_failures_recorded": 0,
        "validated_at_utc": utc_now(),
        "validation_scope": "technical_canary_only",
        "valid_json_responses": completed_metadata["valid_responses"],
    }
    write_json(PROVENANCE_DIR / "canary_validation.json", validation_report)
    update_state("resume_complete", "complete", completed_queries=12)

    artifact_paths = [
        results_path,
        metadata_path,
        STATE_PATH,
        PROVENANCE_DIR / "base_runtime.json",
        PROVENANCE_DIR / "git_identity.json",
        PROVENANCE_DIR / "pip_install.log",
        PROVENANCE_DIR / "pip_check.log",
        PROVENANCE_DIR / "pip_check_report.json",
        PROVENANCE_DIR / "pip_freeze.txt",
        PROVENANCE_DIR / "pinned_package_versions.json",
        PROVENANCE_DIR / "stimulus_transport_verification.json",
        PROVENANCE_DIR / "runner_validate_only.log",
        PROVENANCE_DIR / "frozen_revision_checks.json",
        PROVENANCE_DIR / "colab_preflight.json",
        PROVENANCE_DIR / "invocation_01_max_new_queries_1.log",
        PROVENANCE_DIR / "invocation_02_resume_to_12.log",
        PROVENANCE_DIR / "canary_validation.json",
    ]
    collection = {
        "schema_version": "1.0",
        "attempt_id": _V4_CONTEXT["run_id"],
        "attempt_root": str(ATTEMPT_ROOT),
        "created_at_utc": utc_now(),
        "failures_file": {
            "exists": failures_path.exists(),
            "path": str(failures_path.relative_to(ATTEMPT_ROOT)),
            "size_bytes": failures_path.stat().st_size if failures_path.exists() else 0,
        },
        "files": [
            {
                "path": str(path.relative_to(ATTEMPT_ROOT)),
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in artifact_paths
            if path.is_file()
        ],
        "model_key": MODEL_KEY,
    }
    write_json(PROVENANCE_DIR / "collection_manifest.json", collection)
    print(json.dumps(validation_report, indent=2))
    print(f"Collection manifest: {PROVENANCE_DIR / 'collection_manifest.json'}")
    print("\n===== STEP 5 COMPLETE =====")
    print("TECHNICAL CANARY V4 COMPLETE.")
    print("Do not run full inference before cross-model review and a tracked protocol transition.")
except Exception as error:
    record_failure("resume_complete", "technical_validation", error)
    raise

## Handoff

Success ends with `TECHNICAL CANARY V4 COMPLETE.` Start a fresh Colab runtime and reopen this tagged notebook for the next model.

On failure, do not rerun the cell. Download the notebook with its output and preserve the unique v4 attempt directory for review.